# DynameicEarthNet

Remote sensing is entering a new era of time-series analysis. Short revisit times of satellites allow for monitoring of many areas across the globe on a weekly basis. However, there has been little exploration of deep learning techniques to leverage this new temporal dimension at scale. Especially, existing approaches have struggled to combine the power of different sensors to make use of all available information. In addition, large scale high quality change detection benchmarks are rare. To stimulate innovation in spatio-temporal machine learning, we have partnered up to propose a unique challenge centered around modeling multi-temporal land cover changes from Planetscope and Sentinel time series data.

```
@inproceedings{toker2022dynamicearthnet,
  title={Dynamicearthnet: Daily multi-spectral satellite dataset for semantic change segmentation},
  author={Toker, Aysim and Kondmann, Lukas and Weber, Mark and Eisenberger, Marvin and Camero, Andrés and Hu, Jingliang and Hoderlein, Ariadna Pregel and Senaras, Caglar and Davis, Timothy and Cremers, Daniel and Marchisio, Giovanni and Zhu, Xiao Xiang and Leal-Taixé, Laura},
  booktitle={2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)},
  pages={21126--21135},
  year={2022},
  organization={IEEE}
}
```

The complete dataset is available at: https://mediatum.ub.tum.de/1650201

## 1. Grande-Terre, Guadeloupe

- Planet folder: 20N/27E-76N
- Tile name: 2697_3715_13
- Google Maps: https://maps.app.goo.gl/U347nA1bgk24zVxNA



In [ ]:
from IPython.display import IFrame

IFrame("https://www.google.com/maps/embed?pb=!1m18!1m12!1m3!1d51691.2949337503!2d-61.48382083756777!3d16.483862619942023!2m3!1f0!2f0!3f0!3m2!1i1024!2i768!4f13.1!3m3!1m2!1s0x8c1335dee5121b95%3A0x6baaa206604e9893!2sGrande-Terre!5e1!3m2!1sen!2sde!4v1791463471452!5m2!1sen!2sde", 
       width="100%", height=400)

## 2. Biromaru, Indonesia

- Planet folder: 50S/34E-412N
- Tile name: 6824_4117_13_50S
- Google Maps: https://maps.app.goo.gl/1ttLNxU5EYAKMBuW6


In [ ]:
IFrame("https://www.google.com/maps/embed?pb=!1m18!1m12!1m3!1d17835.880996110664!2d119.89901399056846!3d-0.9615886027399949!2m3!1f0!2f0!3f0!3m2!1i1024!2i768!4f13.1!3m3!1m2!1s0x2d8bf75fadf31657%3A0xca14d076a8f526b9!2sSigi%20Biromaru%2C%20Sigi%20Regency%2C%20Central%20Sulawesi%2C%20Indonesia!5e0!3m2!1sen!2sde!4v1791468642275!5m2!1sen!2sde", 
       width="100%", height=400)

## 3. Data Loader and Transform

In [ ]:
from pathlib import Path
import numpy as np
import rasterio
import h5py

In [ ]:
def load_geotiffs_from_folder(folder_path):
    # Convert string path to Path object and find all .tif / .tiff files
    folder = Path(folder_path)
    tiff_files = sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))
    
    if not tiff_files:
        raise FileNotFoundError(f"No GeoTIFF files found in {folder_path}")
        
    arrays = []
    
    # Loop through and read each file
    for file_path in tiff_files:
        with rasterio.open(file_path) as src:            
            arrays.append(src.read())     
    
    data_cube = np.stack(arrays, axis=0)
    # sort the channels to [sample, height, width, band]
    return np.transpose(data_cube, (0,2,3,1))

In [ ]:
guadeloupe_planet = load_geotiffs_from_folder("raw/guadeloupe/planet")
print("Planet shape:", guadeloupe_planet.shape)

guadeloupe_s1 = load_geotiffs_from_folder("raw/guadeloupe/sentinel-1")
print("S1 shape:", guadeloupe_s1.shape)

guadeloupe_s2 = load_geotiffs_from_folder("raw/guadeloupe/sentinel-2")
print("S2 shape:", guadeloupe_s2.shape)

guadeloupe_labels = load_geotiffs_from_folder("raw/guadeloupe/labels")
print("Labels shape:", guadeloupe_labels.shape)

In [ ]:
biromaru_planet = load_geotiffs_from_folder("raw/biromaru/planet")
print("Planet shape:", biromaru_planet.shape)

biromaru_s1 = load_geotiffs_from_folder("raw/biromaru/sentinel-1")
print("S1 shape:", biromaru_s1.shape)

biromaru_s2 = load_geotiffs_from_folder("raw/biromaru/sentinel-2")
print("S2 shape:", biromaru_s2.shape)

biromaru_labels = load_geotiffs_from_folder("raw/biromaru/labels")
print("Labels shape:", biromaru_labels.shape)

In [ ]:
with h5py.File('dynamic_earth_data.h5', 'w') as hf:
    hf.create_dataset('guadeloupe_planet', data=guadeloupe_planet)
    hf.create_dataset('guadeloupe_s1', data=guadeloupe_s1)
    hf.create_dataset('guadeloupe_s2', data=guadeloupe_s2)
    hf.create_dataset('guadeloupe_labels', data=guadeloupe_labels)

    hf.create_dataset('biromaru_planet', data=biromaru_planet)
    hf.create_dataset('biromaru_s1', data=biromaru_s1)
    hf.create_dataset('biromaru_s2', data=biromaru_s2)
    hf.create_dataset('biromaru_labels', data=biromaru_labels)